# Stage 6 - Split

Runs **before** feature engineering. Train and validation come from the LLM weak labels, split by conversation within each source; **test is the human gold set only**; transfer is the hand-labelled own logs (if present). A leakage check stops the run if any conversation lands in two splits. Settings: `configs/split.yaml`.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[clean]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.split import make_splits

report = make_splits("split")   # writes splits/{train,validation,test,transfer}.jsonl + split_report.json

In [ ]:
import pandas as pd

pd.DataFrame({n: {k: report[n][k] for k in ("segments", "conversations")} for n in ("train", "validation", "test", "transfer")}).T

In [ ]:
pd.DataFrame({n: report[n]["by_source"] for n in ("train", "validation", "test")}).fillna(0).astype(int)

In [ ]:
# Label balance per split: train and validation should look alike; test shows the human view
pd.DataFrame({n: report[n]["knowledge_type_share"] for n in ("train", "validation", "test")})

In [ ]:
pd.DataFrame({n: report[n]["domain_share"] for n in ("train", "validation", "test")}).fillna(0)